In [ ]:
#data loading and training
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

ROOT = Path(r"C:\Users\User\Documents\radar\selected_patches")

CLASS_TO_LABEL = {
    "branch": 0,
    "man": 1,
    "metal": 2,
    "plastic": 3,
    "rock": 4,
}

#prepare dataset

class RadarFusionDataset(Dataset):
    def __init__(self, root):
        self.samples = []

        for class_name, label in CLASS_TO_LABEL.items():
            class_dir = root / class_name
            if not class_dir.exists():
                continue

            for file in class_dir.glob("*.npz"):
                self.samples.append((file, label))

        print(f"Total samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        data = np.load(file_path, allow_pickle=True)

        # ---- rd_patch: (16,16)
        rd = data["rd_patch"].astype(np.float32)
        rd_min, rd_max = rd.min(), rd.max()
        if rd_max > rd_min:
            rd = (rd - rd_min) / (rd_max - rd_min)
        else:
            rd = np.zeros_like(rd, dtype=np.float32)
        rd = np.expand_dims(rd, axis=0)   # (1,16,16)

        # ---- iq_patch: complex, shape ~ (129,16)
        iq = data["iq_patch"]

        iq_real = np.real(iq).astype(np.float32)
        iq_imag = np.imag(iq).astype(np.float32)

        # normalize each channel separately
        def norm2d(x):
            x_min, x_max = x.min(), x.max()
            if x_max > x_min:
                return (x - x_min) / (x_max - x_min)
            else:
                return np.zeros_like(x, dtype=np.float32)

        iq_real = norm2d(iq_real)
        iq_imag = norm2d(iq_imag)

        iq_2ch = np.stack([iq_real, iq_imag], axis=0)   # (2,129,16)

        rd = torch.tensor(rd, dtype=torch.float32)
        iq_2ch = torch.tensor(iq_2ch, dtype=torch.float32)
        y = torch.tensor(label, dtype=torch.long)

        return rd, iq_2ch, y

#RDCNN model

class RDCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # ---- RD branch: input (1,16,16)
        self.rd_branch = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 16->8

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 8->4

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((2, 2))
        )

        # ---- IQ branch: input (2,129,16)
        self.iq_branch = nn.Sequential(
            nn.Conv2d(2, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),   # (129,16)->(64,8)

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),   # -> (32,4)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 2))
        )

        self.classifier = nn.Sequential(
            nn.Linear(64*2*2 + 64*4*2, 128),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(32, 5)
        )

    def forward(self, rd, iq):
        rd_feat = self.rd_branch(rd)
        iq_feat = self.iq_branch(iq)

        rd_feat = torch.flatten(rd_feat, 1)
        iq_feat = torch.flatten(iq_feat, 1)

        feat = torch.cat([rd_feat, iq_feat], dim=1)
        out = self.classifier(feat)
        return out

from torch.utils.data import Subset
from collections import defaultdict, Counter
import random

dataset = RadarFusionDataset(ROOT)

random.seed(42)

TEST_COUNT_PER_CLASS = 30  


class_to_indices = defaultdict(list)
for idx, (_, label) in enumerate(dataset.samples):
    class_to_indices[label].append(idx)

train_indices = []
test_indices = []

for label, indices in class_to_indices.items():
    indices = indices.copy()
    random.shuffle(indices)

    if len(indices) <= TEST_COUNT_PER_CLASS:
        raise ValueError(
            f"Class {label} has only {len(indices)} samples, "
            f"cannot allocate {TEST_COUNT_PER_CLASS} to test."
        )

    test_idx = indices[:TEST_COUNT_PER_CLASS]
    train_idx = indices[TEST_COUNT_PER_CLASS:]

    test_indices.extend(test_idx)
    train_indices.extend(train_idx)

train_dataset = Subset(dataset, train_indices)
test_dataset = Subset(dataset, test_indices)

label_to_class = {v: k for k, v in CLASS_TO_LABEL.items()}

train_labels = [dataset.samples[i][1] for i in train_indices]
test_labels  = [dataset.samples[i][1] for i in test_indices]

print("\nTrain distribution:")
for label, count in sorted(Counter(train_labels).items()):
    print(f"{label_to_class[label]:>8s}: {count}")

print("\nTest distribution:")
for label, count in sorted(Counter(test_labels).items()):
    print(f"{label_to_class[label]:>8s}: {count}")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


model = RDCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=8e-4, weight_decay=1e-4)

#training
num_epochs = 60
best_acc = 0.0
best_epoch = 0

for epoch in range(num_epochs):
    # ---- train
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for rd, iq, y in train_loader:
        rd = rd.to(device)
        iq = iq.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        out = model(rd, iq)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * y.size(0)
        preds = out.argmax(dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ---- test
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for rd, iq, y in test_loader:
            rd = rd.to(device)
            iq = iq.to(device)
            y = y.to(device)

            out = model(rd, iq)
            loss = criterion(out, y)

            test_loss += loss.item() * y.size(0)
            preds = out.argmax(dim=1)
            test_correct += (preds == y).sum().item()
            test_total += y.size(0)

    test_loss /= test_total
    test_acc = test_correct / test_total

    if test_acc > best_acc:
        best_acc = test_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "best_radar_fusion_model.pth")

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}"
    )

print(f"\nBest Test Acc: {best_acc:.4f} at epoch {best_epoch}")
print("Best model saved")

In [ ]:
#test trying(single data)
import numpy as np
import torch
from pathlib import Path


LABEL_TO_CLASS = {
    0: "branch",
    1: "man",
    2: "metal",
    3: "plastic",
    4: "rock",
}

#loading model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RDCNN()  
model.load_state_dict(torch.load("best_radar_fusion_model.pth", map_location=device))
model.to(device)
model.eval()

#prepare testing data

def preprocess_npz(file_path):
    data = np.load(file_path, allow_pickle=True)

    # ---- RD
    rd = data["rd_patch"].astype(np.float32)
    rd_min, rd_max = rd.min(), rd.max()
    if rd_max > rd_min:
        rd = (rd - rd_min) / (rd_max - rd_min)
    else:
        rd = np.zeros_like(rd, dtype=np.float32)
    rd = np.expand_dims(rd, axis=0)  # (1,16,16)

    # ---- IQ
    iq = data["iq_patch"]

    iq_real = np.real(iq).astype(np.float32)
    iq_imag = np.imag(iq).astype(np.float32)

    def norm2d(x):
        x_min, x_max = x.min(), x.max()
        if x_max > x_min:
            return (x - x_min) / (x_max - x_min)
        else:
            return np.zeros_like(x, dtype=np.float32)

    iq_real = norm2d(iq_real)
    iq_imag = norm2d(iq_imag)

    iq = np.stack([iq_real, iq_imag], axis=0)  # (2,129,16)

    rd = torch.tensor(rd, dtype=torch.float32).unsqueeze(0)   # (1,1,16,16)
    iq = torch.tensor(iq, dtype=torch.float32).unsqueeze(0)   # (1,2,129,16)

    return rd.to(device), iq.to(device)

#predict

def predict(file_path):
    rd, iq = preprocess_npz(file_path)

    with torch.no_grad():
        out = model(rd, iq)
        prob = torch.softmax(out, dim=1)
        pred = torch.argmax(prob, dim=1).item()

    print("Prediction:", LABEL_TO_CLASS[pred])
    print("Probabilities:", prob.cpu().numpy())

#test

file_path = r"C:\Users\User\Documents\radar\selected_patches\plastic\plastic_frame000135_sel000.npz"
predict(file_path)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


# load model
model.load_state_dict(torch.load("best_radar_fusion_model.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for rd, iq, y in test_loader:
        rd = rd.to(device)
        iq = iq.to(device)

        out = model(rd, iq)
        preds = out.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(y.numpy())

# confusion matrix
cm = confusion_matrix(all_labels, all_preds)

disp = ConfusionMatrixDisplay(cm, display_labels=["branch", "man","metal","umbrella","rock"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# loading model
model.load_state_dict(torch.load("best_radar_fusion_model.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for rd, iq, y in test_loader:
        rd = rd.to(device)
        iq = iq.to(device)

        out = model(rd, iq)
        preds = out.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(y.numpy())

# =========================
# confusion matrix（%）
# =========================
cm = confusion_matrix(all_labels, all_preds)

cm_percent = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100

disp = ConfusionMatrixDisplay(
    cm_percent,
    display_labels=["branch", "man", "metal", "umbrella", "rock"]
)

disp.plot(cmap="Blues", values_format=".1f")
plt.title("Confusion Matrix (%)")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

ROOT = Path(r"C:\Users\User\Documents\radar\new_data_p\selected_patches")

CLASS_NAMES = ["branch", "man", "metal", "plastic", "rock"]

DISPLAY_NAMES = {
    "branch": "branch",
    "man": "man",
    "metal": "metal",
    "plastic": "umbrella",
    "rock": "rock",
}

def show_all_classes(root, num=6):
    n_class = len(CLASS_NAMES)

    fig, axes = plt.subplots(n_class, num, figsize=(3*num, 2.5*n_class))

    for row, class_name in enumerate(CLASS_NAMES):
        files = list((root / class_name).glob("*.npz"))

        if len(files) == 0:
            print(f"⚠️ {class_name} nodata")
            continue

        files = files[:num]

        for col, file in enumerate(files):
            data = np.load(file, allow_pickle=True)
            rd = data["rd_patch"]

            axes[row, col].imshow(rd, cmap='jet')
            axes[row, col].set_title(DISPLAY_NAMES[class_name])
            axes[row, col].axis("off")

    plt.suptitle("Range-Doppler Examples (All Classes)", fontsize=16)
    plt.tight_layout()
    plt.show()

show_all_classes(ROOT, num=6)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

ROOT = Path(r"C:\Users\User\Documents\radar\new_data_p\selected_patches")
SAVE_DIR = ROOT / "visualization"
SAVE_DIR.mkdir(exist_ok=True)

CLASS_NAMES = ["branch", "man", "metal", "plastic", "rock"]

DISPLAY_NAMES = {
    "branch": "branch",
    "man": "man",
    "metal": "metal",
    "plastic": "umbrella",
    "rock": "rock",
}

def save_per_class_images(root, num=3):
    for class_name in CLASS_NAMES:

        files = list((root / class_name).glob("*.npz"))

        if len(files) == 0:
            print(f"⚠️ {class_name} nodata")
            continue

        files = files[:num]

        fig, axes = plt.subplots(1, num, figsize=(3*num, 3))

        if num == 1:
            axes = [axes]

        for i, file in enumerate(files):
            data = np.load(file, allow_pickle=True)
            rd = data["rd_patch"]

            axes[i].imshow(rd, cmap="jet")
            axes[i].set_title(DISPLAY_NAMES[class_name])
            axes[i].axis("off")

        plt.suptitle(f"{DISPLAY_NAMES[class_name]} examples", fontsize=14)

        save_path = SAVE_DIR / f"{DISPLAY_NAMES[class_name]}.png"
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.close()

        print(f"✅ Saved: {save_path}")

save_per_class_images(ROOT, num=3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path(r"C:\Users\User\Documents\radar\new_data_p\selected_patches")

CLASS_NAMES = ["branch", "man", "metal", "plastic", "rock"]

DISPLAY_NAMES = {
    "branch": "branch",
    "man": "man",
    "metal": "metal",
    "plastic": "umbrella",
    "rock": "rock",
}

def compute_average_rd(root, max_samples=None):
    avg_maps = {}

    for cls in CLASS_NAMES:
        files = list((root / cls).glob("*.npz"))

        if max_samples:
            files = files[:max_samples]

        rd_list = []

        for f in files:
            data = np.load(f, allow_pickle=True)
            rd = data["rd_patch"].astype(np.float32)

            #normalize
            rd = (rd - rd.min()) / (rd.max() - rd.min() + 1e-8)

            rd_list.append(rd)

        rd_stack = np.stack(rd_list, axis=0)
        avg_maps[cls] = np.mean(rd_stack, axis=0)

    return avg_maps

def plot_average_rd(avg_maps):
    n = len(avg_maps)

    fig, axes = plt.subplots(1, n, figsize=(3*n, 3))

    for i, (cls, rd_avg) in enumerate(avg_maps.items()):
        axes[i].imshow(rd_avg, cmap="inferno")
        axes[i].set_title(DISPLAY_NAMES[cls])
        axes[i].axis("off")

    plt.suptitle("Average RD Pattern per Class", fontsize=16)
    plt.tight_layout()
    plt.show()


avg_maps = compute_average_rd(ROOT)
plot_average_rd(avg_maps)